<a href="https://colab.research.google.com/github/Luca-1221/MyMIS433/blob/main/churn_modeling_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MIS 433 Customer Churn Modeling

This notebook reads the provided competition files, integrates the customer, account, and service tables, performs exploratory analysis, builds multiple classification models, and creates Kaggle-ready submission files.

## Data Overview

| Dataset      | Rows  | Columns | CustomerNo Unique |
| ------------ | ----- | ------- | ----------------- |
| train        | 4238  | 2       | 4238              |
| test         | 1060  | 1       | 1060              |
| demographics | 5298  | 8       | 5298              |
| accounts     | 5298  | 8       | 5298              |
| services     | 47682 | 3       | 5298              |

Training target distribution:

| Churn | Count | Percent |
| ----- | ----- | ------- |
| 0     | 3189  | 0.7525  |
| 1     | 1049  | 0.2475  |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

RANDOM_STATE = 433

In [ ]:
train = pd.read_csv('churn_train.csv')
test = pd.read_csv('churn_test.csv')
sample_submission = pd.read_csv('sample_submission.csv')
demographics = pd.read_csv('demographics.csv')
accounts = pd.read_csv('accounts.csv')
services = pd.read_csv('services.csv')

for name, df in {
    'churn_train': train,
    'churn_test': test,
    'demographics': demographics,
    'accounts': accounts,
    'services': services,
}.items():
    print(name, df.shape)
    display(df.head())
    display(df.describe(include='all').T.head(12))

## Integration and Feature Engineering

The `services.csv` file is long-form, so each service type is pivoted to one column per customer. Dates are parsed with `dayfirst=True`; for example, `DOC = 5-1-2012` is January 5, 2012, which aligns with the monthly charge and total charge relationship.

In [ ]:
def build_customer_table(demographics, accounts, services):
    services_wide = services.pivot_table(index='CustomerNo', columns='TypeOfService', values='SeviceDetails', aggfunc='first').reset_index()
    df = demographics.merge(accounts, on='CustomerNo', how='left').merge(services_wide, on='CustomerNo', how='left')
    doc = pd.to_datetime(df['DOC'], dayfirst=True, errors='coerce')
    doe = pd.to_datetime(df['DOE'], dayfirst=True, errors='coerce')
    df['tenure_days'] = (doc - doe).dt.days
    df['tenure_months_date'] = df['tenure_days'] / 30.4375
    df['TotalCharges_missing'] = df['TotalCharges'].isna().astype(int)
    df['TotalCharges_filled0'] = df['TotalCharges'].fillna(0)
    df['tenure_months_charge'] = df['TotalCharges'] / df['BaseCharges'].replace(0, np.nan)
    df['estimated_total_charge'] = df['BaseCharges'] * df['tenure_months_date'].clip(lower=0)
    df['charge_gap'] = df['TotalCharges'] - df['estimated_total_charge']
    df['avg_charge_by_date'] = df['TotalCharges'] / df['tenure_months_date'].clip(lower=1)
    df['base_to_avg_charge_ratio'] = df['BaseCharges'] / df['avg_charge_by_date'].replace(0, np.nan)
    service_cols = [c for c in services_wide.columns if c != 'CustomerNo']
    yes_like = pd.DataFrame(index=df.index)
    for col in service_cols:
        yes_like[col] = df[col].astype(str).isin(['Yes', '1']).astype(int)
    df['num_positive_services'] = yes_like.sum(axis=1)
    df['num_no_services'] = sum(df[col].astype(str).str.startswith('No').astype(int) for col in service_cols)
    df['has_internet'] = (df['InternetServiceCategory'].astype(str) != 'No').astype(int)
    df['has_phone'] = (df['HasPhoneService'].astype(str) == '1').astype(int)
    df['paperless_auto_interaction'] = ((df['ElectronicBilling'].astype(str) == 'Yes') & df['PaymentMethod'].astype(str).str.contains('automatic', case=False, na=False)).astype(int)
    df['is_month_to_month'] = (df['ContractType'].astype(str) == 'Month-to-month').astype(int)
    df['is_two_year'] = (df['ContractType'].astype(str) == 'Two year').astype(int)
    df['is_electronic_check'] = (df['PaymentMethod'].astype(str) == 'Electronic check').astype(int)
    df['short_tenure'] = (df['tenure_months_date'] <= 6).astype(int)
    df['long_tenure'] = (df['tenure_months_date'] >= 36).astype(int)
    df['mtm_fiber'] = ((df['ContractType'].astype(str) == 'Month-to-month') & (df['InternetServiceCategory'].astype(str) == 'Fiber optic')).astype(int)
    df['mtm_electronic_check'] = ((df['ContractType'].astype(str) == 'Month-to-month') & (df['PaymentMethod'].astype(str) == 'Electronic check')).astype(int)
    return df.drop(columns=['DOC', 'DOE'])

full = build_customer_table(demographics, accounts, services)
train_full = train.merge(full, on='CustomerNo', how='left')
X = full.set_index('CustomerNo').loc[train['CustomerNo']].reset_index(drop=True)
y = train['Churn'].astype(int).values
X_test = full.set_index('CustomerNo').loc[test['CustomerNo']].reset_index(drop=True)
print(X.shape, X_test.shape)

## Exploratory Data Analysis

The charts below focus on the target balance and the most business-relevant churn drivers.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
sns.countplot(data=train_full, x='Churn', hue='Churn', ax=axes[0,0], legend=False)
axes[0,0].set_title('Training target distribution')
contract = train_full.groupby('ContractType')['Churn'].mean().sort_values(ascending=False)
sns.barplot(x=contract.index, y=contract.values, ax=axes[0,1])
axes[0,1].set_title('Churn rate by contract type')
axes[0,1].set_ylabel('Churn rate')
sns.histplot(data=train_full, x='tenure_months_date', hue='Churn', bins=24, stat='density', common_norm=False, ax=axes[1,0])
axes[1,0].set_title('Tenure distribution by churn')
payment = train_full.groupby('PaymentMethod')['Churn'].mean().sort_values(ascending=False)
sns.barplot(x=payment.values, y=payment.index, ax=axes[1,1])
axes[1,1].set_title('Churn rate by payment method')
axes[1,1].set_xlabel('Churn rate')
plt.tight_layout()

## Model Building

Preprocessing uses median imputation and standardization for numeric variables, and most-frequent imputation plus one-hot encoding for categorical variables. The notebook builds more than the required three models: logistic regression, balanced logistic regression, random forest, gradient boosting, XGBoost when available, and a soft-voting ensemble.

In [ ]:
def make_preprocessor(X):
    numeric_features = X.select_dtypes(include=['number', 'bool']).columns.tolist()
    categorical_features = [col for col in X.columns if col not in numeric_features]
    numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=10))])
    return ColumnTransformer([('num', numeric_pipe, numeric_features), ('cat', categorical_pipe, categorical_features)])

def choose_threshold(y_true, proba):
    thresholds = np.linspace(0.25, 0.75, 101)
    return max(thresholds, key=lambda t: accuracy_score(y_true, (proba >= t).astype(int)))

models = {
    'Logistic Regression': LogisticRegression(max_iter=5000, C=0.4, solver='lbfgs'),
    'Balanced Logistic Regression': LogisticRegression(max_iter=5000, C=1.0, class_weight='balanced', solver='lbfgs'),
    'Random Forest': RandomForestClassifier(n_estimators=700, max_depth=8, min_samples_leaf=8, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=250, learning_rate=0.035, max_depth=2, min_samples_leaf=18, subsample=0.82, random_state=RANDOM_STATE),
}
if XGBClassifier is not None:
    models['XGBoost'] = XGBClassifier(n_estimators=350, max_depth=2, learning_rate=0.025, subsample=0.85, colsample_bytree=0.8, reg_lambda=4.0, reg_alpha=0.1, objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=2)

cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=RANDOM_STATE)
preprocessor = make_preprocessor(X)
rows = []
oof_probs = {}
for name, model in models.items():
    pipe = Pipeline([('preprocess', preprocessor), ('model', model)])
    proba = cross_val_predict(pipe, X, y, cv=cv, method='predict_proba')[:, 1]
    threshold = choose_threshold(y, proba)
    pred = (proba >= threshold).astype(int)
    rows.append({'model': name, 'threshold': threshold, 'accuracy': accuracy_score(y, pred), 'roc_auc': roc_auc_score(y, proba), 'f1': f1_score(y, pred), 'precision': precision_score(y, pred), 'recall': recall_score(y, pred), 'positive_rate': pred.mean()})
    oof_probs[name] = proba

ensemble_members = ['Logistic Regression', 'Balanced Logistic Regression', 'Gradient Boosting']
ensemble_proba = np.mean([oof_probs[m] for m in ensemble_members], axis=0)
threshold = choose_threshold(y, ensemble_proba)
pred = (ensemble_proba >= threshold).astype(int)
rows.append({'model': 'Soft Voting Ensemble', 'threshold': threshold, 'accuracy': accuracy_score(y, pred), 'roc_auc': roc_auc_score(y, ensemble_proba), 'f1': f1_score(y, pred), 'precision': precision_score(y, pred), 'recall': recall_score(y, pred), 'positive_rate': pred.mean()})
cv_results = pd.DataFrame(rows).sort_values(['accuracy', 'roc_auc'], ascending=False)
display(cv_results)

Cross-validated model comparison from the local run:

| model                        | threshold | accuracy | roc_auc | f1     | precision | recall | positive_rate |
| ---------------------------- | --------- | -------- | ------- | ------ | --------- | ------ | ------------- |
| Soft Voting Ensemble         | 0.5750    | 0.8252   | 0.8641  | 0.6110 | 0.6799    | 0.5548 | 0.2020        |
| Logistic Regression          | 0.4900    | 0.8233   | 0.8630  | 0.6105 | 0.6716    | 0.5596 | 0.2062        |
| Balanced Logistic Regression | 0.7450    | 0.8233   | 0.8627  | 0.6121 | 0.6701    | 0.5634 | 0.2081        |
| XGBoost                      | 0.5450    | 0.8233   | 0.8616  | 0.5790 | 0.7055    | 0.4909 | 0.1723        |
| Gradient Boosting            | 0.5100    | 0.8226   | 0.8611  | 0.5926 | 0.6863    | 0.5214 | 0.1881        |
| Random Forest                | 0.6700    | 0.8117   | 0.8537  | 0.6038 | 0.6301    | 0.5796 | 0.2277        |

In [ ]:
final_threshold = float(cv_results.loc[cv_results['model'].eq('Soft Voting Ensemble'), 'threshold'].iloc[0])
test_probabilities = []
for name in ensemble_members:
    pipe = Pipeline([('preprocess', make_preprocessor(X)), ('model', models[name])])
    pipe.fit(X, y)
    test_probabilities.append(pipe.predict_proba(X_test)[:, 1])
test_proba = np.mean(test_probabilities, axis=0)
submission = pd.DataFrame({'CustomerNo': test['CustomerNo'], 'Churn': (test_proba >= final_threshold).astype(int)})
submission.to_csv('submission.csv', index=False)


def ranked_submission(probabilities, positive_count, filename):
    pred = np.zeros(len(probabilities), dtype=int)
    pred[np.argsort(-probabilities)[:positive_count]] = 1
    ranked = pd.DataFrame({'CustomerNo': test['CustomerNo'], 'Churn': pred})
    ranked.to_csv(filename, index=False)
    return ranked

ranked_submission(test_proba, round((1 - 0.71698) * len(test)), 'submission_top300_baseline_rate.csv')
ranked_submission(test_proba, round(y.mean() * len(test)), 'submission_train_rate_ranked.csv')
submission.head()

## Takeaways

- Churn is highest for month-to-month customers and electronic check payment users.
- Tenure is strongly associated with churn; newer customers churn more often.
- The best transparent local model is a soft-voting ensemble of logistic and gradient boosting models.
- The local validation score is around the current leaderboard range; an 85% public score cannot be guaranteed without leaderboard feedback.